# 🎬🎵📚 Bilingual Recommendation Chatbot
## Complete Notebook: Data → NLP Model → Chatbot Engine

**Dataset**: Custom intent training data (bilingual ID + EN)  
**Tujuan**: Membangun chatbot rekomendasi film, musik, dan buku  

---
| Part | Isi |
|---|---|
| **Part 1** | Data & Intent Design |
| **Part 2** | NLP Model Training (TF-IDF + ML) |
| **Part 3** | Chatbot Engine & Demo |

> 💡 Jalankan semua cell dari atas ke bawah secara berurutan

## 1. Install & Import Libraries

In [ ]:
!pip install nltk langdetect pandas scikit-learn matplotlib seaborn -q

import json
import re
import string
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer
from langdetect import detect

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})
BLUE   = '#0052D9'
ORANGE = '#FF6B35'
GREEN  = '#1DB954'
RED    = '#FF3B6B'

print('✅ Semua library berhasil diimport!')

## 2. Desain Intent & Training Data

Chatbot ini mengenali **9 intent** dalam dua bahasa (Indonesia & Inggris).

| Intent | Deskripsi |
|---|---|
| `greet` | Sapaan pembuka |
| `recommend_movie` | Minta rekomendasi film |
| `recommend_music` | Minta rekomendasi musik/lagu |
| `recommend_book` | Minta rekomendasi buku |
| `filter_genre` | Filter berdasarkan genre |
| `filter_mood` | Filter berdasarkan mood/suasana |
| `ask_more` | Minta rekomendasi tambahan |
| `thanks` | Ucapan terima kasih |
| `goodbye` | Pamitan |

In [ ]:
# Dataset training intent — kalimat dalam Bahasa Indonesia dan Inggris
intents_data = [

    # ── GREET ─────────────────────────────────────────────────────────────
    {'intent': 'greet', 'text': 'halo', 'lang': 'id'},
    {'intent': 'greet', 'text': 'hai', 'lang': 'id'},
    {'intent': 'greet', 'text': 'hei', 'lang': 'id'},
    {'intent': 'greet', 'text': 'selamat pagi', 'lang': 'id'},
    {'intent': 'greet', 'text': 'selamat siang', 'lang': 'id'},
    {'intent': 'greet', 'text': 'selamat malam', 'lang': 'id'},
    {'intent': 'greet', 'text': 'apa kabar', 'lang': 'id'},
    {'intent': 'greet', 'text': 'halo bot', 'lang': 'id'},
    {'intent': 'greet', 'text': 'hey', 'lang': 'en'},
    {'intent': 'greet', 'text': 'hello', 'lang': 'en'},
    {'intent': 'greet', 'text': 'hi there', 'lang': 'en'},
    {'intent': 'greet', 'text': 'good morning', 'lang': 'en'},
    {'intent': 'greet', 'text': 'good afternoon', 'lang': 'en'},
    {'intent': 'greet', 'text': 'good evening', 'lang': 'en'},
    {'intent': 'greet', 'text': 'how are you', 'lang': 'en'},
    {'intent': 'greet', 'text': 'whats up', 'lang': 'en'},
    {'intent': 'greet', 'text': 'hey bot', 'lang': 'en'},

    # ── RECOMMEND MOVIE ───────────────────────────────────────────────────
    {'intent': 'recommend_movie', 'text': 'rekomendasiin film dong', 'lang': 'id'},
    {'intent': 'recommend_movie', 'text': 'film apa yang bagus', 'lang': 'id'},
    {'intent': 'recommend_movie', 'text': 'mau nonton film apa ya', 'lang': 'id'},
    {'intent': 'recommend_movie', 'text': 'kasih saran film', 'lang': 'id'},
    {'intent': 'recommend_movie', 'text': 'film apa yang recommended', 'lang': 'id'},
    {'intent': 'recommend_movie', 'text': 'pengen nonton tapi bingung', 'lang': 'id'},
    {'intent': 'recommend_movie', 'text': 'film terbaik apa', 'lang': 'id'},
    {'intent': 'recommend_movie', 'text': 'suggest me a movie', 'lang': 'en'},
    {'intent': 'recommend_movie', 'text': 'recommend a film please', 'lang': 'en'},
    {'intent': 'recommend_movie', 'text': 'what movie should i watch', 'lang': 'en'},
    {'intent': 'recommend_movie', 'text': 'i want to watch something', 'lang': 'en'},
    {'intent': 'recommend_movie', 'text': 'good movies to watch', 'lang': 'en'},
    {'intent': 'recommend_movie', 'text': 'best movies recommendation', 'lang': 'en'},
    {'intent': 'recommend_movie', 'text': 'any movie suggestions', 'lang': 'en'},
    {'intent': 'recommend_movie', 'text': 'what should i watch tonight', 'lang': 'en'},

    # ── RECOMMEND MUSIC ───────────────────────────────────────────────────
    {'intent': 'recommend_music', 'text': 'rekomendasiin lagu dong', 'lang': 'id'},
    {'intent': 'recommend_music', 'text': 'kasih playlist', 'lang': 'id'},
    {'intent': 'recommend_music', 'text': 'lagu apa yang enak didengar', 'lang': 'id'},
    {'intent': 'recommend_music', 'text': 'musik apa yang bagus', 'lang': 'id'},
    {'intent': 'recommend_music', 'text': 'pengen dengerin musik apa ya', 'lang': 'id'},
    {'intent': 'recommend_music', 'text': 'kasih rekomendasi musik', 'lang': 'id'},
    {'intent': 'recommend_music', 'text': 'lagu yang cocok buat belajar', 'lang': 'id'},
    {'intent': 'recommend_music', 'text': 'suggest me a song', 'lang': 'en'},
    {'intent': 'recommend_music', 'text': 'recommend some music', 'lang': 'en'},
    {'intent': 'recommend_music', 'text': 'what song should i listen to', 'lang': 'en'},
    {'intent': 'recommend_music', 'text': 'good songs to listen', 'lang': 'en'},
    {'intent': 'recommend_music', 'text': 'music recommendation please', 'lang': 'en'},
    {'intent': 'recommend_music', 'text': 'give me a playlist', 'lang': 'en'},
    {'intent': 'recommend_music', 'text': 'what music do you suggest', 'lang': 'en'},
    {'intent': 'recommend_music', 'text': 'any good tracks', 'lang': 'en'},

    # ── RECOMMEND BOOK ────────────────────────────────────────────────────
    {'intent': 'recommend_book', 'text': 'rekomendasiin buku dong', 'lang': 'id'},
    {'intent': 'recommend_book', 'text': 'buku apa yang menarik', 'lang': 'id'},
    {'intent': 'recommend_book', 'text': 'mau baca buku apa ya', 'lang': 'id'},
    {'intent': 'recommend_book', 'text': 'kasih saran buku', 'lang': 'id'},
    {'intent': 'recommend_book', 'text': 'buku yang recommended', 'lang': 'id'},
    {'intent': 'recommend_book', 'text': 'pengen baca tapi bingung', 'lang': 'id'},
    {'intent': 'recommend_book', 'text': 'buku bagus apa', 'lang': 'id'},
    {'intent': 'recommend_book', 'text': 'suggest me a book', 'lang': 'en'},
    {'intent': 'recommend_book', 'text': 'recommend a book please', 'lang': 'en'},
    {'intent': 'recommend_book', 'text': 'what book should i read', 'lang': 'en'},
    {'intent': 'recommend_book', 'text': 'good books to read', 'lang': 'en'},
    {'intent': 'recommend_book', 'text': 'best book recommendations', 'lang': 'en'},
    {'intent': 'recommend_book', 'text': 'any book suggestions', 'lang': 'en'},
    {'intent': 'recommend_book', 'text': 'i want to read something interesting', 'lang': 'en'},

    # ── FILTER GENRE ──────────────────────────────────────────────────────
    {'intent': 'filter_genre', 'text': 'yang genre action', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'yang horror', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'yang romance', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'genre comedy dong', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'yang sci-fi', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'drama korea', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'yang thriller', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'yang pop', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'yang jazz', 'lang': 'id'},
    {'intent': 'filter_genre', 'text': 'something action', 'lang': 'en'},
    {'intent': 'filter_genre', 'text': 'i like horror', 'lang': 'en'},
    {'intent': 'filter_genre', 'text': 'romantic genre please', 'lang': 'en'},
    {'intent': 'filter_genre', 'text': 'comedy movies', 'lang': 'en'},
    {'intent': 'filter_genre', 'text': 'sci-fi recommendation', 'lang': 'en'},
    {'intent': 'filter_genre', 'text': 'thriller please', 'lang': 'en'},
    {'intent': 'filter_genre', 'text': 'something in pop genre', 'lang': 'en'},
    {'intent': 'filter_genre', 'text': 'jazz music', 'lang': 'en'},

    # ── FILTER MOOD ───────────────────────────────────────────────────────
    {'intent': 'filter_mood', 'text': 'lagi sedih nih', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'lagi senang banget', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'pengen yang santai', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'butuh semangat', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'lagi galau', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'mood lagi down', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'yang bikin ketawa', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'yang menegangkan', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'yang romantis', 'lang': 'id'},
    {'intent': 'filter_mood', 'text': 'im feeling sad', 'lang': 'en'},
    {'intent': 'filter_mood', 'text': 'i am happy today', 'lang': 'en'},
    {'intent': 'filter_mood', 'text': 'something relaxing', 'lang': 'en'},
    {'intent': 'filter_mood', 'text': 'need motivation', 'lang': 'en'},
    {'intent': 'filter_mood', 'text': 'feeling down', 'lang': 'en'},
    {'intent': 'filter_mood', 'text': 'something funny', 'lang': 'en'},
    {'intent': 'filter_mood', 'text': 'exciting and thrilling', 'lang': 'en'},
    {'intent': 'filter_mood', 'text': 'something romantic', 'lang': 'en'},

    # ── ASK MORE ──────────────────────────────────────────────────────────
    {'intent': 'ask_more', 'text': 'ada lagi', 'lang': 'id'},
    {'intent': 'ask_more', 'text': 'kasih yang lain dong', 'lang': 'id'},
    {'intent': 'ask_more', 'text': 'rekomendasiin yang lain', 'lang': 'id'},
    {'intent': 'ask_more', 'text': 'lagi dong', 'lang': 'id'},
    {'intent': 'ask_more', 'text': 'masih ada rekomendasi lain', 'lang': 'id'},
    {'intent': 'ask_more', 'text': 'give me more', 'lang': 'en'},
    {'intent': 'ask_more', 'text': 'any other suggestions', 'lang': 'en'},
    {'intent': 'ask_more', 'text': 'show me more', 'lang': 'en'},
    {'intent': 'ask_more', 'text': 'what else do you have', 'lang': 'en'},
    {'intent': 'ask_more', 'text': 'more please', 'lang': 'en'},

    # ── THANKS ────────────────────────────────────────────────────────────
    {'intent': 'thanks', 'text': 'makasih', 'lang': 'id'},
    {'intent': 'thanks', 'text': 'terima kasih', 'lang': 'id'},
    {'intent': 'thanks', 'text': 'thanks ya', 'lang': 'id'},
    {'intent': 'thanks', 'text': 'mantap', 'lang': 'id'},
    {'intent': 'thanks', 'text': 'oke thanks', 'lang': 'id'},
    {'intent': 'thanks', 'text': 'thank you', 'lang': 'en'},
    {'intent': 'thanks', 'text': 'thanks a lot', 'lang': 'en'},
    {'intent': 'thanks', 'text': 'great thanks', 'lang': 'en'},
    {'intent': 'thanks', 'text': 'awesome thank you', 'lang': 'en'},
    {'intent': 'thanks', 'text': 'cheers', 'lang': 'en'},

    # ── GOODBYE ───────────────────────────────────────────────────────────
    {'intent': 'goodbye', 'text': 'bye', 'lang': 'id'},
    {'intent': 'goodbye', 'text': 'sampai jumpa', 'lang': 'id'},
    {'intent': 'goodbye', 'text': 'dadah', 'lang': 'id'},
    {'intent': 'goodbye', 'text': 'pamit dulu', 'lang': 'id'},
    {'intent': 'goodbye', 'text': 'see you', 'lang': 'en'},
    {'intent': 'goodbye', 'text': 'goodbye', 'lang': 'en'},
    {'intent': 'goodbye', 'text': 'bye bye', 'lang': 'en'},
    {'intent': 'goodbye', 'text': 'take care', 'lang': 'en'},
    {'intent': 'goodbye', 'text': 'catch you later', 'lang': 'en'},
]

df_intents = pd.DataFrame(intents_data)
print(f'✅ Total training samples : {len(df_intents)}')
print(f'✅ Jumlah intent          : {df_intents["intent"].nunique()}')
print(f'✅ Bahasa Indonesia       : {(df_intents["lang"]=="id").sum()} kalimat')
print(f'✅ Bahasa Inggris         : {(df_intents["lang"]=="en").sum()} kalimat')
display(df_intents.head(10))

## 3. Preprocessing Teks

Tiga langkah preprocessing sebelum teks bisa diproses model:

| Langkah | Contoh |
|---|---|
| **Lowercasing** | "Rekomendasiin Film" → "rekomendasiin film" |
| **Stopword removal** | "film apa yang bagus" → "film bagus" |
| **Stemming** | "rekomendasiin" → "rekomendasi" |

In [ ]:
# Install PySastrawi dulu sebelum import
!pip install PySastrawi -q

import importlib, sys
importlib.invalidate_caches()

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# Setup stemmer
stemmer_en  = PorterStemmer()
stemmer_id  = StemmerFactory().create_stemmer()

# Stopwords
stop_en = set(stopwords.words('english'))
stop_id = set(stopwords.words('indonesian'))
stop_id.update(['dong', 'nih', 'ya', 'deh', 'sih', 'yuk', 'lah', 'banget', 'aja'])
stop_en.update(['please', 'me', 'give', 'i', 'want', 'need'])

def preprocess(text, lang='id'):
    """Preprocessing: lowercase → hapus tanda baca → tokenize → stopword → stem"""
    text   = text.lower()
    text   = re.sub(r'[^\w\s]', ' ', text)
    text   = re.sub(r'\d+', '', text)
    tokens = word_tokenize(text)
    stops  = stop_id if lang == 'id' else stop_en
    tokens = [t for t in tokens if t not in stops and len(t) > 1]
    if lang == 'id':
        # Sastrawi stem seluruh string sekaligus
        return stemmer_id.stem(' '.join(tokens))
    else:
        return ' '.join([stemmer_en.stem(t) for t in tokens])

# Terapkan preprocessing
df_intents['text_clean'] = df_intents.apply(
    lambda row: preprocess(row['text'], row['lang']), axis=1
)

print('📊 Perbandingan sebelum vs sesudah preprocessing:\n')
sample = df_intents.sample(8, random_state=42)[['text', 'lang', 'text_clean', 'intent']]
display(sample)

In [ ]:
# Deteksi bahasa otomatis dengan langdetect
def detect_lang(text):
    try:
        detected = detect(text)
        return 'id' if detected == 'id' else 'en'
    except:
        return 'en'  # default ke English jika gagal detect

# Test deteksi bahasa
test_sentences = [
    ('rekomendasiin film horror dong', 'id'),
    ('suggest me a relaxing book', 'en'),
    ('lagi sedih butuh musik yang enak', 'id'),
    ('what movie should i watch tonight', 'en'),
    ('kasih playlist buat belajar', 'id'),
]

print('🔍 Test Language Detection:\n')
print(f'{"Kalimat":<45} {"Aktual":<8} {"Deteksi":<8} {"Status"}')
print('-' * 70)
for text, actual in test_sentences:
    detected = detect_lang(text)
    status   = '✅' if detected == actual else '❌'
    print(f'{text:<45} {actual:<8} {detected:<8} {status}')

## 4. Visualisasi Distribusi Intent

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Jumlah sampel per intent
intent_counts = df_intents['intent'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(intent_counts)))
bars = axes[0].barh(intent_counts.index, intent_counts.values,
                    color=colors, alpha=0.85)
axes[0].set_title('Jumlah Sampel per Intent', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Jumlah Kalimat')
for bar, val in zip(bars, intent_counts.values):
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=9)

# Plot 2: Distribusi bahasa
lang_counts = df_intents['lang'].value_counts()
axes[1].pie(lang_counts.values,
            labels=['Bahasa Indonesia', 'Bahasa Inggris'],
            colors=[BLUE, ORANGE], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Distribusi Bahasa', fontsize=12, fontweight='bold')

# Plot 3: Distribusi bahasa per intent
pivot = df_intents.groupby(['intent', 'lang']).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=axes[2], color=[BLUE, ORANGE], alpha=0.8, width=0.7)
axes[2].set_title('Distribusi Bahasa per Intent', fontsize=12, fontweight='bold')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=30)
axes[2].legend(['Indonesia', 'English'], fontsize=9)

plt.suptitle('Analisis Training Data Intent', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📊 Total training samples : {len(df_intents)}')
print(f'   Rata-rata per intent   : {len(df_intents)/df_intents["intent"].nunique():.1f} kalimat')

In [ ]:
# Visualisasi panjang kalimat
df_intents['text_len']       = df_intents['text'].apply(lambda x: len(x.split()))
df_intents['text_clean_len'] = df_intents['text_clean'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_intents['text_len'], bins=15, color=BLUE, alpha=0.75, edgecolor='none')
axes[0].set_title('Distribusi Panjang Kalimat (Asli)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(df_intents['text_len'].mean(), color=ORANGE, linestyle='--',
                label=f"Mean = {df_intents['text_len'].mean():.1f}")
axes[0].legend()

axes[1].hist(df_intents['text_clean_len'], bins=15, color=GREEN, alpha=0.75, edgecolor='none')
axes[1].set_title('Distribusi Panjang Kalimat (Setelah Preprocessing)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Jumlah Kata')
axes[1].axvline(df_intents['text_clean_len'].mean(), color=ORANGE, linestyle='--',
                label=f"Mean = {df_intents['text_clean_len'].mean():.1f}")
axes[1].legend()

plt.suptitle('Panjang Kalimat Sebelum vs Sesudah Preprocessing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Membangun Database Rekomendasi

Database berisi koleksi film, musik, dan buku yang akan direkomendasikan chatbot.
Setiap item memiliki **genre** dan **mood tag** sehingga bisa difilter sesuai preferensi user.

In [ ]:
# ── DATABASE FILM ─────────────────────────────────────────────────────────
movies_data = [
    # Action
    {'title': 'Mad Max: Fury Road', 'genre': 'action', 'mood': ['exciting', 'intense'], 'rating': 8.1, 'year': 2015, 'desc': 'Post-apocalyptic chase thriller'},
    {'title': 'John Wick', 'genre': 'action', 'mood': ['exciting', 'intense'], 'rating': 7.4, 'year': 2014, 'desc': 'Retired hitman seeks revenge'},
    {'title': 'Top Gun: Maverick', 'genre': 'action', 'mood': ['exciting', 'motivating'], 'rating': 8.3, 'year': 2022, 'desc': 'Elite pilot faces new mission'},
    {'title': 'Mission Impossible: Fallout', 'genre': 'action', 'mood': ['exciting', 'intense'], 'rating': 7.7, 'year': 2018, 'desc': 'Ethan Hunt saves the world again'},
    # Horror
    {'title': 'Get Out', 'genre': 'horror', 'mood': ['tense', 'intense'], 'rating': 7.7, 'year': 2017, 'desc': 'Racial horror thriller'},
    {'title': 'A Quiet Place', 'genre': 'horror', 'mood': ['tense', 'intense'], 'rating': 7.5, 'year': 2018, 'desc': 'Family survives in silence'},
    {'title': 'Hereditary', 'genre': 'horror', 'mood': ['tense', 'sad'], 'rating': 7.3, 'year': 2018, 'desc': 'Family haunted by dark ancestry'},
    {'title': 'The Conjuring', 'genre': 'horror', 'mood': ['tense', 'intense'], 'rating': 7.5, 'year': 2013, 'desc': 'Paranormal investigators help a family'},
    # Romance
    {'title': 'La La Land', 'genre': 'romance', 'mood': ['happy', 'sad', 'relaxing'], 'rating': 8.0, 'year': 2016, 'desc': 'Love story between two dreamers'},
    {'title': 'The Notebook', 'genre': 'romance', 'mood': ['sad', 'happy'], 'rating': 7.8, 'year': 2004, 'desc': 'Epic love story across decades'},
    {'title': 'Crazy Rich Asians', 'genre': 'romance', 'mood': ['happy', 'relaxing'], 'rating': 6.9, 'year': 2018, 'desc': 'Romance meets Asian family drama'},
    # Comedy
    {'title': 'The Grand Budapest Hotel', 'genre': 'comedy', 'mood': ['happy', 'relaxing'], 'rating': 8.1, 'year': 2014, 'desc': 'Quirky adventures of a hotel concierge'},
    {'title': 'Superbad', 'genre': 'comedy', 'mood': ['happy', 'funny'], 'rating': 7.6, 'year': 2007, 'desc': 'High school friends last adventure'},
    {'title': 'Parasite', 'genre': 'comedy', 'mood': ['intense', 'exciting'], 'rating': 8.5, 'year': 2019, 'desc': 'Class divide dark comedy thriller'},
    # Sci-Fi
    {'title': 'Interstellar', 'genre': 'sci-fi', 'mood': ['exciting', 'sad', 'motivating'], 'rating': 8.6, 'year': 2014, 'desc': 'Journey through space and time'},
    {'title': 'Dune', 'genre': 'sci-fi', 'mood': ['exciting', 'intense'], 'rating': 8.0, 'year': 2021, 'desc': 'Epic science fiction saga'},
    {'title': 'Everything Everywhere All at Once', 'genre': 'sci-fi', 'mood': ['happy', 'sad', 'funny'], 'rating': 7.8, 'year': 2022, 'desc': 'Multiverse adventure comedy'},
    # Drama
    {'title': 'The Shawshank Redemption', 'genre': 'drama', 'mood': ['motivating', 'sad'], 'rating': 9.3, 'year': 1994, 'desc': 'Hope and friendship in prison'},
    {'title': 'Forrest Gump', 'genre': 'drama', 'mood': ['happy', 'sad', 'motivating'], 'rating': 8.8, 'year': 1994, 'desc': 'Life journey of a simple man'},
    {'title': 'Soul', 'genre': 'drama', 'mood': ['relaxing', 'motivating', 'happy'], 'rating': 8.0, 'year': 2020, 'desc': 'Jazz musician discovers purpose of life'},
    # Thriller
    {'title': 'Gone Girl', 'genre': 'thriller', 'mood': ['intense', 'tense'], 'rating': 8.1, 'year': 2014, 'desc': 'Mystery behind a missing wife'},
    {'title': 'Knives Out', 'genre': 'thriller', 'mood': ['exciting', 'funny'], 'rating': 7.9, 'year': 2019, 'desc': 'Whodunit mystery comedy'},
    {'title': 'Prisoners', 'genre': 'thriller', 'mood': ['intense', 'tense', 'sad'], 'rating': 8.1, 'year': 2013, 'desc': 'Father searches for missing daughters'},
]

df_movies = pd.DataFrame(movies_data)
print(f'🎬 Film: {len(df_movies)} judul, {df_movies["genre"].nunique()} genre')
display(df_movies.head(5))

In [ ]:
# ── DATABASE MUSIK ────────────────────────────────────────────────────────
music_data = [
    # Pop
    {'title': 'Blinding Lights', 'artist': 'The Weeknd', 'genre': 'pop', 'mood': ['happy', 'exciting'], 'year': 2019},
    {'title': 'Levitating', 'artist': 'Dua Lipa', 'genre': 'pop', 'mood': ['happy', 'exciting'], 'year': 2020},
    {'title': 'Anti-Hero', 'artist': 'Taylor Swift', 'genre': 'pop', 'mood': ['relaxing', 'sad'], 'year': 2022},
    {'title': 'As It Was', 'artist': 'Harry Styles', 'genre': 'pop', 'mood': ['sad', 'relaxing'], 'year': 2022},
    {'title': 'Stay', 'artist': 'The Kid LAROI', 'genre': 'pop', 'mood': ['happy', 'exciting'], 'year': 2021},
    # Rock
    {'title': 'Bohemian Rhapsody', 'artist': 'Queen', 'genre': 'rock', 'mood': ['exciting', 'happy'], 'year': 1975},
    {'title': 'Smells Like Teen Spirit', 'artist': 'Nirvana', 'genre': 'rock', 'mood': ['intense', 'exciting'], 'year': 1991},
    {'title': 'Hotel California', 'artist': 'Eagles', 'genre': 'rock', 'mood': ['relaxing', 'sad'], 'year': 1977},
    # Jazz
    {'title': 'Take Five', 'artist': 'Dave Brubeck', 'genre': 'jazz', 'mood': ['relaxing', 'happy'], 'year': 1959},
    {'title': 'So What', 'artist': 'Miles Davis', 'genre': 'jazz', 'mood': ['relaxing', 'sad'], 'year': 1959},
    {'title': 'Fly Me to the Moon', 'artist': 'Frank Sinatra', 'genre': 'jazz', 'mood': ['happy', 'relaxing'], 'year': 1964},
    # Lo-fi
    {'title': 'Snowfall', 'artist': 'Øneheart & reidenshi', 'genre': 'lofi', 'mood': ['relaxing', 'sad'], 'year': 2021},
    {'title': 'Chill Lofi Study Beats', 'artist': 'Lofi Girl', 'genre': 'lofi', 'mood': ['relaxing', 'focusing'], 'year': 2020},
    # R&B
    {'title': 'Golden', 'artist': 'Harry Styles', 'genre': 'rnb', 'mood': ['happy', 'exciting'], 'year': 2020},
    {'title': 'Superstar', 'artist': 'Usher', 'genre': 'rnb', 'mood': ['happy', 'exciting'], 'year': 2004},
    # Electronic
    {'title': 'One More Time', 'artist': 'Daft Punk', 'genre': 'electronic', 'mood': ['happy', 'exciting'], 'year': 2000},
    {'title': 'Strobe', 'artist': 'deadmau5', 'genre': 'electronic', 'mood': ['relaxing', 'sad'], 'year': 2009},
    # Motivational
    {'title': "Don't Stop Believin", 'artist': 'Journey', 'genre': 'rock', 'mood': ['motivating', 'happy'], 'year': 1981},
    {'title': 'Eye of the Tiger', 'artist': 'Survivor', 'genre': 'rock', 'mood': ['motivating', 'exciting'], 'year': 1982},
    {'title': 'Happy', 'artist': 'Pharrell Williams', 'genre': 'pop', 'mood': ['happy', 'motivating'], 'year': 2013},
    # Sad
    {'title': 'The Night We Met', 'artist': 'Lord Huron', 'genre': 'indie', 'mood': ['sad', 'relaxing'], 'year': 2014},
    {'title': 'someone like you', 'artist': 'Adele', 'genre': 'pop', 'mood': ['sad', 'relaxing'], 'year': 2011},
]

df_music = pd.DataFrame(music_data)
print(f'🎵 Musik: {len(df_music)} lagu, {df_music["genre"].nunique()} genre')
display(df_music.head(5))

In [ ]:
# ── DATABASE BUKU ─────────────────────────────────────────────────────────
books_data = [
    # Self-help / Motivasi
    {'title': 'Atomic Habits', 'author': 'James Clear', 'genre': 'self-help', 'mood': ['motivating', 'focusing'], 'rating': 4.4, 'desc': 'Build good habits, break bad ones'},
    {'title': 'The Alchemist', 'author': 'Paulo Coelho', 'genre': 'self-help', 'mood': ['motivating', 'relaxing'], 'rating': 4.2, 'desc': 'Follow your dreams journey'},
    {'title': 'Ikigai', 'author': 'Hector Garcia', 'genre': 'self-help', 'mood': ['relaxing', 'motivating'], 'rating': 4.0, 'desc': 'Japanese secret to long happy life'},
    {'title': 'The Power of Now', 'author': 'Eckhart Tolle', 'genre': 'self-help', 'mood': ['relaxing', 'motivating'], 'rating': 4.1, 'desc': 'Living in the present moment'},
    # Fiction
    {'title': 'The Midnight Library', 'author': 'Matt Haig', 'genre': 'fiction', 'mood': ['sad', 'motivating', 'relaxing'], 'rating': 4.2, 'desc': 'Library between life and death'},
    {'title': 'Project Hail Mary', 'author': 'Andy Weir', 'genre': 'fiction', 'mood': ['exciting', 'motivating'], 'rating': 4.5, 'desc': 'Astronaut saves earth alone'},
    {'title': 'The Martian', 'author': 'Andy Weir', 'genre': 'fiction', 'mood': ['exciting', 'funny'], 'rating': 4.4, 'desc': 'Stranded astronaut survives on Mars'},
    {'title': 'Tomorrow and Tomorrow', 'author': 'Gabrielle Zevin', 'genre': 'fiction', 'mood': ['sad', 'happy', 'relaxing'], 'rating': 4.3, 'desc': 'Love and video games'},
    # Thriller / Mystery
    {'title': 'Gone Girl', 'author': 'Gillian Flynn', 'genre': 'thriller', 'mood': ['intense', 'exciting'], 'rating': 4.0, 'desc': 'Missing wife mystery thriller'},
    {'title': 'The Girl with the Dragon Tattoo', 'author': 'Stieg Larsson', 'genre': 'thriller', 'mood': ['intense', 'exciting'], 'rating': 4.1, 'desc': 'Journalist investigates old mystery'},
    {'title': 'Big Little Lies', 'author': 'Liane Moriarty', 'genre': 'thriller', 'mood': ['intense', 'funny'], 'rating': 4.1, 'desc': 'Secrets among school parents'},
    # Romance
    {'title': 'The Hating Game', 'author': 'Sally Thorne', 'genre': 'romance', 'mood': ['happy', 'funny'], 'rating': 3.9, 'desc': 'Office rivals fall in love'},
    {'title': 'Beach Read', 'author': 'Emily Henry', 'genre': 'romance', 'mood': ['happy', 'relaxing'], 'rating': 3.9, 'desc': 'Two writers and a summer bet'},
    # Non-fiction
    {'title': 'Sapiens', 'author': 'Yuval Noah Harari', 'genre': 'non-fiction', 'mood': ['motivating', 'exciting'], 'rating': 4.4, 'desc': 'Brief history of humankind'},
    {'title': 'Educated', 'author': 'Tara Westover', 'genre': 'non-fiction', 'mood': ['motivating', 'intense'], 'rating': 4.5, 'desc': 'Self-educated girl escapes family'},
    {'title': 'Thinking Fast and Slow', 'author': 'Daniel Kahneman', 'genre': 'non-fiction', 'mood': ['focusing', 'relaxing'], 'rating': 4.2, 'desc': 'How we think and make decisions'},
    # Fantasy
    {'title': 'The Name of the Wind', 'author': 'Patrick Rothfuss', 'genre': 'fantasy', 'mood': ['exciting', 'relaxing'], 'rating': 4.5, 'desc': 'Legend of the most infamous magician'},
    {'title': 'The Way of Kings', 'author': 'Brandon Sanderson', 'genre': 'fantasy', 'mood': ['exciting', 'motivating'], 'rating': 4.6, 'desc': 'Epic fantasy saga'},
]

df_books = pd.DataFrame(books_data)
print(f'📚 Buku: {len(df_books)} judul, {df_books["genre"].nunique()} genre')
display(df_books.head(5))

## 6. Analisis Database Rekomendasi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Film per genre
movie_genre = df_movies['genre'].value_counts()
axes[0].bar(movie_genre.index, movie_genre.values, color=BLUE, alpha=0.8)
axes[0].set_title('🎬 Film per Genre', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=30)

# Musik per genre
music_genre = df_music['genre'].value_counts()
axes[1].bar(music_genre.index, music_genre.values, color=ORANGE, alpha=0.8)
axes[1].set_title('🎵 Musik per Genre', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Genre')
axes[1].tick_params(axis='x', rotation=30)

# Buku per genre
book_genre = df_books['genre'].value_counts()
axes[2].bar(book_genre.index, book_genre.values, color=GREEN, alpha=0.8)
axes[2].set_title('📚 Buku per Genre', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Genre')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Distribusi Genre Database Rekomendasi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('📊 Ringkasan Database:')
print(f'  🎬 Film  : {len(df_movies)} judul, {df_movies["genre"].nunique()} genre, {df_movies["mood"].explode().nunique()} mood tags')
print(f'  🎵 Musik : {len(df_music)} lagu, {df_music["genre"].nunique()} genre, {df_music["mood"].explode().nunique()} mood tags')
print(f'  📚 Buku  : {len(df_books)} judul, {df_books["genre"].nunique()} genre, {df_books["mood"].explode().nunique()} mood tags')

## 7. Simpan Data

In [ ]:
# Simpan training data intent
df_intents.to_csv('intent_data.csv', index=False)
print('💾 intent_data.csv tersimpan')

# Simpan database rekomendasi
df_movies.to_json('db_movies.json', orient='records', indent=2)
df_music.to_json('db_music.json', orient='records', indent=2)
df_books.to_json('db_books.json', orient='records', indent=2)
print('💾 db_movies.json, db_music.json, db_books.json tersimpan')

print('\n=' * 55)
print('✅ RINGKASAN NOTEBOOK 1')
print('=' * 55)
print(f'''
TRAINING DATA:
  Total kalimat  : {len(df_intents)}
  Jumlah intent  : {df_intents['intent'].nunique()}
  Bahasa         : Indonesia + Inggris

DATABASE REKOMENDASI:
  Film           : {len(df_movies)} judul
  Musik          : {len(df_music)} lagu
  Buku           : {len(df_books)} judul

OUTPUT FILE:
  ✅ intent_data.csv
  ✅ db_movies.json
  ✅ db_music.json
  ✅ db_books.json
''')
print('🎉 Notebook 1 selesai! Lanjut → Notebook 2: NLP Model Training')

---
# 🤖 Part 2: NLP Model Training
> Lanjutan Part 1 — pastikan file intent_data.csv sudah tersimpan

## 1. Import & Load Data

In [ ]:
!pip install PySastrawi langdetect -q

import re
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from sklearn.pipeline import Pipeline

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})
BLUE   = '#0052D9'
ORANGE = '#FF6B35'
GREEN  = '#1DB954'
RED    = '#FF3B6B'

print('✅ Semua library berhasil diimport!')

In [ ]:
# Load data intent dari Notebook 1
df = pd.read_csv('intent_data.csv')

print(f'✅ Data dimuat: {len(df)} kalimat, {df["intent"].nunique()} intent')
print(f'   Bahasa Indonesia : {(df["lang"]=="id").sum()} kalimat')
print(f'   Bahasa Inggris   : {(df["lang"]=="en").sum()} kalimat')
display(df.head(8))

## 2. Persiapan Fitur & Label

In [ ]:
# Rebuild preprocessing
stemmer_en = PorterStemmer()
stemmer_id = StemmerFactory().create_stemmer()
stop_en    = set(stopwords.words('english'))
stop_id    = set(stopwords.words('indonesian'))
stop_id.update(['dong', 'nih', 'ya', 'deh', 'sih', 'yuk', 'lah', 'banget', 'aja'])
stop_en.update(['please', 'me', 'give', 'i', 'want', 'need'])

def preprocess(text, lang='id'):
    text   = str(text).lower()
    text   = re.sub(r'[^\w\s]', ' ', text)
    text   = re.sub(r'\d+', '', text)
    tokens = word_tokenize(text)
    stops  = stop_id if lang == 'id' else stop_en
    tokens = [t for t in tokens if t not in stops and len(t) > 1]
    if not tokens:
        return str(text).strip()  # fallback ke teks asli
    if lang == 'id':
        return stemmer_id.stem(' '.join(tokens))
    return ' '.join([stemmer_en.stem(t) for t in tokens])

# Preprocessing ulang
if 'text_clean' not in df.columns:
    df['text_clean'] = df.apply(lambda r: preprocess(r['text'], r['lang']), axis=1)

# Bersihkan NaN — pakai masking, bukan .replace()
df['text_clean'] = df['text_clean'].fillna(df['text'])
mask = df['text_clean'].str.strip() == ''
df.loc[mask, 'text_clean'] = df.loc[mask, 'text']
df = df[df['text_clean'].notna()].reset_index(drop=True)

# Fitur dan label
X = df['text_clean'].astype(str).values
y = df['intent'].values

intent_labels = sorted(df['intent'].unique())
label2idx     = {l: i for i, l in enumerate(intent_labels)}
idx2label     = {i: l for l, i in label2idx.items()}
y_enc         = np.array([label2idx[l] for l in y])

print(f'✅ Fitur dan label siap')
print(f'   X shape : {X.shape}  (tidak ada NaN)')
print(f'   y shape : {y_enc.shape}')
print(f'   Intent  : {intent_labels}')

In [ ]:
# Train-test split (80:20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Pastikan semua string valid
X_train = np.array([str(x) for x in X_train])
X_test  = np.array([str(x) for x in X_test])

print(f'📊 Pembagian data:')
print(f'   Training : {len(X_train)} kalimat')
print(f'   Test     : {len(X_test)} kalimat')

## 3. TF-IDF Vectorization

**TF-IDF** (*Term Frequency-Inverse Document Frequency*) mengubah teks menjadi vektor angka.

- **TF** → seberapa sering kata muncul dalam satu kalimat
- **IDF** → seberapa unik kata tersebut di seluruh dataset
- Kata yang sering muncul di banyak kalimat (misal 'yang') → bobot rendah
- Kata yang spesifik (misal 'film', 'buku') → bobot tinggi

In [ ]:
# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),  # unigram + bigram
    max_features=500,    # ambil 500 fitur terpenting
    sublinear_tf=True    # log scaling untuk TF
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'✅ TF-IDF selesai')
print(f'   Vocab size      : {len(tfidf.vocabulary_)} kata/frasa')
print(f'   X_train shape   : {X_train_tfidf.shape}')
print(f'   X_test shape    : {X_test_tfidf.shape}')

# Tampilkan top 20 kata dengan bobot TF-IDF tertinggi
feature_names = tfidf.get_feature_names_out()
tfidf_scores  = X_train_tfidf.toarray().mean(axis=0)
top20_idx     = tfidf_scores.argsort()[-20:][::-1]

fig, ax = plt.subplots(figsize=(12, 4))
ax.barh([feature_names[i] for i in top20_idx],
        [tfidf_scores[i] for i in top20_idx],
        color=BLUE, alpha=0.8)
ax.set_title('Top 20 Kata/Frasa dengan Bobot TF-IDF Tertinggi',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Rata-rata TF-IDF Score')
plt.tight_layout()
plt.show()

## 4. Model 1: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)
acc_lr    = accuracy_score(y_test, y_pred_lr)
f1_lr     = f1_score(y_test, y_pred_lr, average='weighted')

print('📊 Logistic Regression:')
print(f'   Accuracy : {acc_lr*100:.2f}%')
print(f'   F1 Score : {f1_lr:.4f}')

## 5. Model 2: Support Vector Machine (SVM)

In [ ]:
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm.fit(X_train_tfidf, y_train)

y_pred_svm = svm.predict(X_test_tfidf)
acc_svm    = accuracy_score(y_test, y_pred_svm)
f1_svm     = f1_score(y_test, y_pred_svm, average='weighted')

print('📊 Support Vector Machine:')
print(f'   Accuracy : {acc_svm*100:.2f}%')
print(f'   F1 Score : {f1_svm:.4f}')

## 6. Model 3: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)

y_pred_rf = rf.predict(X_test_tfidf)
acc_rf    = accuracy_score(y_test, y_pred_rf)
f1_rf     = f1_score(y_test, y_pred_rf, average='weighted')

print('📊 Random Forest:')
print(f'   Accuracy : {acc_rf*100:.2f}%')
print(f'   F1 Score : {f1_rf:.4f}')

## 7. Perbandingan Model

In [ ]:
results = pd.DataFrame({
    'Model'   : ['Logistic Regression', 'SVM (LinearSVC)', 'Random Forest'],
    'Accuracy': [acc_lr, acc_svm, acc_rf],
    'F1 Score': [f1_lr, f1_svm, f1_rf],
})
results['Accuracy (%)'] = (results['Accuracy'] * 100).round(2)
results['F1 Score']     = results['F1 Score'].round(4)

print('=' * 55)
print('📊 PERBANDINGAN MODEL')
print('=' * 55)
display(results[['Model','Accuracy (%)','F1 Score']])

best_model_name = results.loc[results['Accuracy'].idxmax(), 'Model']
best_acc        = results['Accuracy'].max()
print(f'\n🏆 Model terbaik: {best_model_name} (Accuracy = {best_acc*100:.2f}%)')

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors    = [BLUE, ORANGE, GREEN]

for ax, metric, col in zip(axes, ['Accuracy (%)', 'F1 Score'], colors):
    bars = ax.bar(results['Model'], results[metric], color=colors, alpha=0.8, width=0.5)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(results[metric].min() * 0.9, results[metric].max() * 1.05)
    ax.tick_params(axis='x', rotation=10)
    for bar, val in zip(bars, results[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val}', ha='center', fontsize=10, fontweight='bold')
    # Highlight best
    best_idx = results[metric].idxmax()
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2)

plt.suptitle('Perbandingan Model Intent Classifier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Evaluasi Detail — Model Terbaik

Confusion matrix dan classification report untuk model dengan akurasi tertinggi.

In [ ]:
# Pilih model terbaik otomatis
model_map  = {
    'Logistic Regression': (lr, y_pred_lr),
    'SVM (LinearSVC)'    : (svm, y_pred_svm),
    'Random Forest'      : (rf, y_pred_rf),
}
best_name  = results.loc[results['Accuracy'].idxmax(), 'Model']
best_model, y_pred_best = model_map[best_name]
print(f'Model terbaik: {best_name}\n')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Absolut
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=intent_labels, yticklabels=intent_labels,
            ax=axes[0], linewidths=0.5, linecolor='white')
axes[0].set_title(f'Confusion Matrix — {best_name}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Prediksi', fontsize=10)
axes[0].set_ylabel('Aktual', fontsize=10)
axes[0].tick_params(axis='x', rotation=35)

# Persentase
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=intent_labels, yticklabels=intent_labels,
            ax=axes[1], linewidths=0.5, linecolor='white', vmin=0, vmax=100)
axes[1].set_title('Confusion Matrix (%) per Kelas', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Prediksi', fontsize=10)
axes[1].set_ylabel('Aktual', fontsize=10)
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
report = classification_report(
    y_test, y_pred_best,
    target_names=intent_labels,
    output_dict=True
)
report_df = pd.DataFrame(report).T.round(3)
print(f'📊 Classification Report — {best_name}:')
display(report_df)

# F1 per intent
f1_per_intent = [report[intent]['f1-score'] for intent in intent_labels]

fig, ax = plt.subplots(figsize=(11, 4))
colors_bar = [GREEN if f >= 0.9 else ORANGE if f >= 0.7 else RED for f in f1_per_intent]
bars = ax.bar(intent_labels, f1_per_intent, color=colors_bar, alpha=0.85)
ax.axhline(np.mean(f1_per_intent), color='black', linestyle='--',
           label=f'Rata-rata F1 = {np.mean(f1_per_intent):.3f}')
ax.set_title('F1-Score per Intent', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=10)
ax.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, f1_per_intent):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 9. Cross Validation

Cross validation memberikan gambaran akurasi yang lebih stabil
— tidak bergantung pada satu split tertentu.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('⏳ Menjalankan 5-Fold Cross Validation...\n')
cv_results = {}

for name, model in [('Logistic Regression', lr),
                     ('SVM', svm),
                     ('Random Forest', rf)]:
    scores = cross_val_score(model, X_train_tfidf, y_train,
                             cv=skf, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name}:')
    print(f'  Scores : {[round(s, 4) for s in scores]}')
    print(f'  Mean   : {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%\n')

# Visualisasi boxplot CV
fig, ax = plt.subplots(figsize=(10, 4))
bp = ax.boxplot(
    [cv_results[k] for k in cv_results],
    labels=list(cv_results.keys()),
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], [BLUE, ORANGE, GREEN]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('5-Fold Cross Validation Accuracy', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

## 10. Simpan Model Terbaik

In [ ]:
# Buat pipeline lengkap: TF-IDF + model terbaik
# Pipeline mempermudah prediksi — cukup input teks mentah

# Tentukan model terbaik
best_clf = {'Logistic Regression': lr, 'SVM (LinearSVC)': svm,
             'Random Forest': rf}[best_name]

# Fit ulang dengan seluruh data (train+test) untuk hasil optimal
tfidf_full = TfidfVectorizer(ngram_range=(1, 2), max_features=500, sublinear_tf=True)

best_pipeline = Pipeline([
    ('tfidf', tfidf_full),
    ('clf',   best_clf.__class__(**best_clf.get_params()))
])
best_pipeline.fit(X, y_enc)

# Simpan pipeline
with open('intent_pipeline.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)

# Simpan label mapping
with open('label_mapping.json', 'w') as f:
    json.dump({'label2idx': label2idx, 'idx2label': {str(k): v
               for k, v in idx2label.items()}}, f, indent=2)

print(f'💾 intent_pipeline.pkl tersimpan ({best_name})')
print(f'💾 label_mapping.json tersimpan')

# Test pipeline dengan kalimat baru
print('\n🧪 Test prediksi pipeline:')
test_sentences = [
    'rekomendasiin film action dong',
    'suggest me a relaxing book',
    'lagi sedih butuh lagu yang enak',
    'what music should i listen to',
    'ada rekomendasi lagi',
    'thank you so much',
]
for sent in test_sentences:
    pred_idx   = best_pipeline.predict([sent])[0]
    pred_label = idx2label[pred_idx]
    print(f'  "{sent}" → {pred_label}')

In [ ]:
print('=' * 55)
print('✅ RINGKASAN NOTEBOOK 2')
print('=' * 55)
print(f'''
MODEL TERBAIK : {best_name}
Accuracy      : {results.loc[results["Accuracy"].idxmax(), "Accuracy (%)"]:.2f}%
F1 Score      : {results.loc[results["F1 Score"].idxmax(), "F1 Score"]:.4f}

PIPELINE:
  Input  → Teks mentah (bahasa apa pun)
  Step 1 → TF-IDF Vectorizer (ngram 1-2, 500 fitur)
  Step 2 → {best_name} Classifier
  Output → Predicted Intent Label

OUTPUT FILE:
  ✅ intent_pipeline.pkl  ← model siap dipakai chatbot
  ✅ label_mapping.json   ← mapping index ke nama intent
''')
print('🎉 Notebook 2 selesai! Lanjut → Notebook 3: Chatbot Engine')

---
# 💬 Part 3: Chatbot Engine & Demo
> Lanjutan Part 2 — pastikan intent_pipeline.pkl dan db_*.json sudah tersimpan

## 1. Import & Load Semua Aset

In [ ]:
!pip install PySastrawi langdetect -q

import re
import json
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, LangDetectException

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

BLUE   = '#0052D9'
ORANGE = '#FF6B35'
GREEN  = '#1DB954'
RED    = '#FF3B6B'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})
print('✅ Libraries loaded!')

In [ ]:
# Load model pipeline & label mapping
with open('intent_pipeline.pkl', 'rb') as f:
    pipeline = pickle.load(f)

with open('label_mapping.json', 'r') as f:
    mapping  = json.load(f)
    idx2label = {int(k): v for k, v in mapping['idx2label'].items()}
    label2idx = mapping['label2idx']

# Load database rekomendasi
with open('db_movies.json', 'r') as f:
    movies = json.load(f)
with open('db_music.json', 'r') as f:
    music  = json.load(f)
with open('db_books.json', 'r') as f:
    books  = json.load(f)

print('✅ Semua aset berhasil dimuat!')
print(f'   Intent pipeline : {type(pipeline).__name__}')
print(f'   Intent labels   : {list(label2idx.keys())}')
print(f'   Database film   : {len(movies)} judul')
print(f'   Database musik  : {len(music)} lagu')
print(f'   Database buku   : {len(books)} judul')

## 2. Language Detector

Deteksi bahasa otomatis — menentukan apakah user menulis dalam Bahasa Indonesia atau Inggris.

In [ ]:
def detect_language(text):
    """Deteksi bahasa: return 'id' atau 'en'"""
    try:
        lang = detect(text)
        return 'id' if lang == 'id' else 'en'
    except LangDetectException:
        return 'en'  # default English

# Test
test_cases = [
    ('rekomendasiin film horror dong', 'id'),
    ('suggest me a relaxing book', 'en'),
    ('lagi sedih butuh musik enak', 'id'),
    ('what movie should i watch tonight', 'en'),
    ('halo', 'id'),
    ('hello there', 'en'),
]

print('🔍 Test Language Detection:')
print(f'{"Kalimat":<42} {"Aktual":<8} {"Deteksi":<8} {"Status"}')
print('-' * 65)
for text, actual in test_cases:
    det    = detect_language(text)
    status = '✅' if det == actual else '❌'
    print(f'{text:<42} {actual:<8} {det:<8} {status}')

## 3. Entity Extractor

Mengekstrak **genre** dan **mood** dari kalimat user — digunakan untuk memfilter database rekomendasi.

In [ ]:
# Mapping kata → genre & mood dalam dua bahasa
GENRE_MAP = {
    # Film genres (ID & EN)
    'action': 'action',        'aksi': 'action',
    'horror': 'horror',        'horor': 'horror',      'seram': 'horror',
    'romance': 'romance',      'romantis': 'romance',  'roman': 'romance',
    'comedy': 'comedy',        'komedi': 'comedy',     'lucu': 'comedy',
    'thriller': 'thriller',    'menegangkan': 'thriller',
    'drama': 'drama',
    'sci-fi': 'sci-fi',        'fiksi': 'sci-fi',      'science': 'sci-fi',
    # Music genres
    'pop': 'pop',
    'rock': 'rock',
    'jazz': 'jazz',
    'lofi': 'lofi',            'lo-fi': 'lofi',        'santai': 'lofi',
    'electronic': 'electronic','elektro': 'electronic',
    'rnb': 'rnb',              'r&b': 'rnb',
    'indie': 'indie',
    # Book genres
    'fantasy': 'fantasy',      'fantasi': 'fantasy',
    'self-help': 'self-help',  'motivasi': 'self-help','pengembangan': 'self-help',
    'non-fiction': 'non-fiction', 'nonfiksi': 'non-fiction',
    'fiction': 'fiction',      'fiksi ilmiah': 'fiction',
}

MOOD_MAP = {
    # Positive
    'happy': 'happy',          'senang': 'happy',      'bahagia': 'happy',
    'funny': 'funny',          'lucu': 'funny',         'ketawa': 'funny',
    'exciting': 'exciting',    'seru': 'exciting',      'semangat': 'exciting',
    'motivating': 'motivating','motivasi': 'motivating','motivate': 'motivating',
    # Negative
    'sad': 'sad',              'sedih': 'sad',          'galau': 'sad',
    'down': 'sad',             'lonely': 'sad',         'patah': 'sad',
    # Neutral
    'relaxing': 'relaxing',    'santai': 'relaxing',    'tenang': 'relaxing',
    'relax': 'relaxing',       'kalem': 'relaxing',
    'intense': 'intense',      'intens': 'intense',
    'tense': 'tense',          'tegang': 'tense',
    'focusing': 'focusing',    'fokus': 'focusing',     'belajar': 'focusing',
}

CONTENT_TYPE_MAP = {
    'film': 'movie',    'movie': 'movie',   'nonton': 'movie',  'watch': 'movie',
    'lagu': 'music',    'musik': 'music',   'music': 'music',   'song': 'music',
    'playlist': 'music','dengerin': 'music','listen': 'music',
    'buku': 'book',     'book': 'book',     'baca': 'book',     'read': 'book',
}

def extract_entities(text):
    """Ekstrak genre, mood, dan content_type dari teks user."""
    text_lower = text.lower()
    words      = re.findall(r'\w+', text_lower)

    genre        = None
    mood         = None
    content_type = None

    for word in words:
        if word in GENRE_MAP and not genre:
            genre = GENRE_MAP[word]
        if word in MOOD_MAP and not mood:
            mood = MOOD_MAP[word]
        if word in CONTENT_TYPE_MAP and not content_type:
            content_type = CONTENT_TYPE_MAP[word]

    return {'genre': genre, 'mood': mood, 'content_type': content_type}

# Test
test_phrases = [
    'rekomendasiin film action yang seru',
    'lagi sedih butuh musik yang santai',
    'suggest me a relaxing book',
    'horror movie please',
    'pengen dengerin jazz',
]
print('🔍 Test Entity Extraction:')
for phrase in test_phrases:
    ent = extract_entities(phrase)
    print(f'  "{phrase}"')
    print(f'    → genre={ent["genre"]}, mood={ent["mood"]}, content_type={ent["content_type"]}')

## 4. Recommendation Engine

In [ ]:
def get_recommendations(content_type, genre=None, mood=None, n=3):
    """Filter database dan return n rekomendasi acak."""
    if content_type == 'movie':
        db = movies
    elif content_type == 'music':
        db = music
    else:
        db = books

    # Filter berdasarkan genre dan/atau mood
    filtered = db
    if genre:
        filtered = [item for item in filtered if item.get('genre') == genre]
    if mood and filtered:
        mood_filtered = [item for item in filtered if mood in item.get('mood', [])]
        # Kalau mood filter terlalu ketat, fallback ke genre filter saja
        filtered = mood_filtered if mood_filtered else filtered

    # Fallback ke seluruh database jika filter terlalu ketat
    if not filtered:
        filtered = db

    # Acak dan ambil n item
    return random.sample(filtered, min(n, len(filtered)))

# Test
print('🎬 Test - Film Action Seru:')
recs = get_recommendations('movie', genre='action', mood='exciting')
for r in recs:
    print(f'  • {r["title"]} ({r.get("year","-")}) — {r.get("desc","")}')

print('\n🎵 Test - Musik Santai:')
recs = get_recommendations('music', mood='relaxing')
for r in recs:
    print(f'  • {r["title"]} — {r["artist"]}')

print('\n📚 Test - Buku Motivasi:')
recs = get_recommendations('book', mood='motivating')
for r in recs:
    print(f'  • {r["title"]} — {r["author"]}')

## 5. Response Generator (Bilingual)

In [ ]:
# Template respons dalam dua bahasa
RESPONSES = {
    'greet': {
        'id': [
            'Halo! Aku siap bantu rekomendasiin film, musik, atau buku buat kamu 🎬🎵📚',
            'Hai! Mau cari rekomendasi apa hari ini? Film, musik, atau buku?',
            'Hei! Aku RecBot, siap kasih rekomendasi terbaik buat kamu!',
        ],
        'en': [
            "Hey there! I can recommend movies, music, or books for you 🎬🎵📚",
            "Hi! Looking for a movie, some music, or a good book?",
            "Hello! I'm RecBot, ready to give you the best recommendations!",
        ]
    },
    'ask_type': {
        'id': 'Mau rekomendasi apa? 🎬 Film, 🎵 Musik, atau 📚 Buku?',
        'en': 'What would you like? 🎬 Movies, 🎵 Music, or 📚 Books?'
    },
    'recommend_movie': {
        'id': 'Nih beberapa film yang mungkin kamu suka 🎬',
        'en': "Here are some movies you might enjoy 🎬"
    },
    'recommend_music': {
        'id': 'Dengerin ini deh, cocok banget! 🎵',
        'en': 'Check these out, you might love them! 🎵'
    },
    'recommend_book': {
        'id': 'Buku-buku ini kayaknya bakal kamu suka 📚',
        'en': 'These books might be perfect for you 📚'
    },
    'ask_more': {
        'id': 'Oke, ini rekomendasi tambahan buat kamu!',
        'en': "Sure! Here are more recommendations for you!"
    },
    'thanks': {
        'id': [
            'Sama-sama! Semoga rekomendasi tadi cocok ya 😊',
            'Senang bisa bantu! Kalau butuh rekomendasi lagi, tanya aja!',
        ],
        'en': [
            "You're welcome! Hope you enjoy the recommendations 😊",
            "Happy to help! Feel free to ask for more anytime!",
        ]
    },
    'goodbye': {
        'id': [
            'Sampai jumpa! Selamat menikmati 👋',
            'Dadah! Semoga harimu menyenangkan 😊',
        ],
        'en': [
            'Goodbye! Enjoy your recommendations 👋',
            'See you! Have a great time 😊',
        ]
    },
    'fallback': {
        'id': 'Hmm, aku kurang ngerti maksudnya. Coba tanya rekomendasi film, musik, atau buku ya!',
        'en': "Hmm, I'm not sure I understand. Try asking for a movie, music, or book recommendation!"
    }
}

def format_recommendations(items, content_type, lang):
    """Format daftar rekomendasi menjadi teks yang rapi."""
    lines = []
    for item in items:
        if content_type == 'movie':
            year  = item.get('year', '')
            desc  = item.get('desc', '')
            rating= item.get('rating', '')
            lines.append(f"🎬 **{item['title']}** ({year}) ⭐{rating}\n   {desc}")
        elif content_type == 'music':
            lines.append(f"🎵 **{item['title']}** — {item['artist']} ({item.get('year','')})")
        else:  # book
            rating = item.get('rating', '')
            desc   = item.get('desc', '')
            lines.append(f"📚 **{item['title']}** — {item['author']} ⭐{rating}\n   {desc}")
    return '\n'.join(lines)

def get_response_text(key, lang):
    """Ambil teks respons, pilih acak jika tersedia beberapa versi."""
    resp = RESPONSES.get(key, RESPONSES['fallback'])
    text = resp.get(lang, resp.get('en', ''))
    if isinstance(text, list):
        return random.choice(text)
    return text

print('✅ Response templates siap!')

## 6. Chatbot Engine

Komponen utama yang menggabungkan semua modul: language detection → intent classification → entity extraction → recommendation → response.

In [ ]:
class RecBot:
    def __init__(self, pipeline, idx2label, movies, music, books):
        self.pipeline   = pipeline
        self.idx2label  = idx2label
        self.db         = {'movie': movies, 'music': music, 'book': books}
        self.context    = {
            'last_content_type': None,
            'last_genre'       : None,
            'last_mood'        : None,
            'last_lang'        : 'id',
        }
        self.history    = []

    def predict_intent(self, text):
        """Prediksi intent dari teks user."""
        pred_idx = self.pipeline.predict([text])[0]
        return self.idx2label[pred_idx]

    def get_recs(self, content_type, genre=None, mood=None, n=3):
        """Ambil rekomendasi dari database."""
        db       = self.db.get(content_type, self.db['movie'])
        filtered = db
        if genre:
            g = [i for i in filtered if i.get('genre') == genre]
            if g: filtered = g
        if mood:
            m = [i for i in filtered if mood in i.get('mood', [])]
            if m: filtered = m
        return random.sample(filtered, min(n, len(filtered)))

    def respond(self, user_input):
        """Proses input user dan kembalikan respons chatbot."""
        # 1. Deteksi bahasa
        lang = detect_language(user_input)
        self.context['last_lang'] = lang

        # 2. Klasifikasi intent
        intent = self.predict_intent(user_input)

        # 3. Ekstrak entitas
        entities = extract_entities(user_input)
        genre    = entities['genre']
        mood     = entities['mood']
        ctype    = entities['content_type']

        # 4. Update konteks dengan entitas baru
        if genre: self.context['last_genre'] = genre
        if mood:  self.context['last_mood']  = mood
        if ctype: self.context['last_content_type'] = ctype

        # 5. Generate respons berdasarkan intent
        response = ''

        if intent == 'greet':
            response = get_response_text('greet', lang)

        elif intent in ['recommend_movie', 'recommend_music', 'recommend_book']:
            # Tentukan content type dari intent
            ctype_map = {
                'recommend_movie': 'movie',
                'recommend_music': 'music',
                'recommend_book' : 'book'
            }
            ctype = ctype_map[intent]
            self.context['last_content_type'] = ctype
            recs  = self.get_recs(ctype, genre, mood)
            intro = get_response_text(intent, lang)
            formatted = format_recommendations(recs, ctype, lang)
            response  = f"{intro}\n\n{formatted}"

        elif intent == 'filter_genre':
            ctype = self.context.get('last_content_type') or 'movie'
            recs  = self.get_recs(ctype, genre=genre,
                                   mood=self.context.get('last_mood'))
            ctype_resp = {
                'movie': get_response_text('recommend_movie', lang),
                'music': get_response_text('recommend_music', lang),
                'book' : get_response_text('recommend_book',  lang)
            }[ctype]
            formatted = format_recommendations(recs, ctype, lang)
            response  = f"{ctype_resp}\n\n{formatted}"

        elif intent == 'filter_mood':
            ctype = self.context.get('last_content_type') or 'movie'
            recs  = self.get_recs(ctype, genre=self.context.get('last_genre'),
                                   mood=mood)
            ctype_resp = {
                'movie': get_response_text('recommend_movie', lang),
                'music': get_response_text('recommend_music', lang),
                'book' : get_response_text('recommend_book',  lang)
            }[ctype]
            formatted  = format_recommendations(recs, ctype, lang)
            response   = f"{ctype_resp}\n\n{formatted}"

        elif intent == 'ask_more':
            ctype = self.context.get('last_content_type') or 'movie'
            recs  = self.get_recs(ctype, self.context.get('last_genre'),
                                   self.context.get('last_mood'))
            intro     = get_response_text('ask_more', lang)
            formatted = format_recommendations(recs, ctype, lang)
            response  = f"{intro}\n\n{formatted}"

        elif intent == 'thanks':
            response = get_response_text('thanks', lang)

        elif intent == 'goodbye':
            response = get_response_text('goodbye', lang)

        else:
            response = get_response_text('fallback', lang)

        # Simpan ke history
        self.history.append({
            'user'  : user_input,
            'intent': intent,
            'lang'  : lang,
            'bot'   : response
        })

        return intent, lang, response

# Inisialisasi chatbot
bot = RecBot(pipeline, idx2label, movies, music, books)
print('✅ RecBot siap digunakan!')

## 7. Demo Percakapan End-to-End

Simulasi percakapan nyata — campuran Bahasa Indonesia dan Inggris.

In [ ]:
def chat_demo(inputs):
    """Jalankan demo percakapan dan tampilkan hasilnya."""
    bot_demo = RecBot(pipeline, idx2label, movies, music, books)
    print('=' * 65)
    print('💬 DEMO PERCAKAPAN')
    print('=' * 65)
    for user_input in inputs:
        intent, lang, response = bot_demo.respond(user_input)
        lang_flag = '🇮🇩' if lang == 'id' else '🇬🇧'
        print(f'\n👤 User  : {user_input}')
        print(f'   [{lang_flag} {lang.upper()} | intent: {intent}]')
        print(f'🤖 RecBot: {response}')
        print('-' * 65)

# Skenario 1: Percakapan Bahasa Indonesia
demo_id = [
    'halo',
    'rekomendasiin film dong',
    'yang genre action',
    'ada lagi?',
    'sekarang rekomendasiin musik',
    'lagi sedih nih',
    'makasih ya!',
    'sampai jumpa',
]
chat_demo(demo_id)

In [ ]:
# Skenario 2: Percakapan Bahasa Inggris
demo_en = [
    'hello there',
    'suggest me a book',
    'something motivating please',
    'more please',
    'now recommend me a movie',
    'i like sci-fi',
    'thanks a lot!',
    'goodbye',
]
chat_demo(demo_en)

In [ ]:
# Skenario 3: Campuran (Code Switching)
demo_mix = [
    'hey',
    'rekomendasiin music dong',
    'something relaxing',
    'give me more',
    'buku apa yang bagus',
    'yang genre thriller',
    'thank you!',
]
chat_demo(demo_mix)

## 8. Evaluasi Chatbot

In [ ]:
# Test akurasi intent classifier pada kalimat-kalimat baru
test_cases = [
    ('mau nonton apa ya malam ini', 'recommend_movie'),
    ('what book should i read', 'recommend_book'),
    ('kasih playlist buat kerja', 'recommend_music'),
    ('yang seram-seram dong', 'filter_genre'),
    ('im feeling really down today', 'filter_mood'),
    ('ada rekomendasi lain', 'ask_more'),
    ('thanks bro', 'thanks'),
    ('dadah', 'goodbye'),
    ('selamat pagi', 'greet'),
    ('good night', 'greet'),
]

correct = 0
print(f'{"Kalimat":<40} {"Aktual":<20} {"Prediksi":<20} {"Status"}')
print('-' * 90)
for text, actual in test_cases:
    predicted = bot.predict_intent(text)
    status    = '✅' if predicted == actual else '❌'
    if predicted == actual: correct += 1
    print(f'{text:<40} {actual:<20} {predicted:<20} {status}')

acc = correct / len(test_cases) * 100
print(f'\n🎯 Akurasi pada test cases baru: {correct}/{len(test_cases)} = {acc:.1f}%')

In [ ]:
# Visualisasi distribusi intent dari history percakapan
# (run setelah demo selesai)
all_history = bot.history
if all_history:
    hist_df = pd.DataFrame(all_history)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Intent distribution
    intent_dist = hist_df['intent'].value_counts()
    colors      = plt.cm.Set3(np.linspace(0, 1, len(intent_dist)))
    axes[0].bar(intent_dist.index, intent_dist.values, color=colors, alpha=0.85)
    axes[0].set_title('Distribusi Intent dalam Percakapan', fontsize=11, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].set_ylabel('Jumlah')

    # Language distribution
    lang_dist = hist_df['lang'].value_counts()
    axes[1].pie(lang_dist.values,
                labels=['Bahasa Indonesia' if l=='id' else 'English' for l in lang_dist.index],
                colors=[BLUE, ORANGE], autopct='%1.0f%%', startangle=90)
    axes[1].set_title('Distribusi Bahasa dalam Percakapan', fontsize=11, fontweight='bold')

    plt.suptitle('Analisis Percakapan Demo', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Simpan Chatbot Engine

In [ ]:
# Simpan seluruh konfigurasi chatbot
chatbot_config = {
    'responses': RESPONSES,
    'genre_map' : GENRE_MAP,
    'mood_map'  : MOOD_MAP,
    'content_type_map': CONTENT_TYPE_MAP,
}

with open('chatbot_config.json', 'w', encoding='utf-8') as f:
    json.dump(chatbot_config, f, ensure_ascii=False, indent=2)

print('💾 File tersimpan:')
print('   chatbot_config.json  ← response templates + entity maps')
print('\n📦 File yang dibutuhkan untuk Streamlit app:')
print('   ✅ intent_pipeline.pkl')
print('   ✅ label_mapping.json')
print('   ✅ chatbot_config.json')
print('   ✅ db_movies.json')
print('   ✅ db_music.json')
print('   ✅ db_books.json')

print('\n=' * 55)
print('🎉 Notebook 3 selesai! Chatbot siap dideploy ke Streamlit!')
print('=' * 55)